In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_samples = 30
factor = rng.normal(size=n_samples)

coord_data = {
    f"COORD_{i}": factor * rng.normal(1.0, 0.15) + rng.normal(scale=0.2, size=n_samples)
    for i in range(5)
}
noise_data = {
    f"NOISE_{i}": rng.normal(size=n_samples)
    for i in range(15)
}

samples = [f"sample_{i}" for i in range(n_samples)]
expression = pd.DataFrame({**coord_data, **noise_data}, index=samples).T
expression.columns = samples

In [4]:
from romapy.core import ROMA
from sklearn.decomposition import PCA

coord_submatrix = expression.loc[[f"COORD_{i}" for i in range(5)]]
samples_list = coord_submatrix.columns.tolist()

leave_one_out_l1 = []
for sample in samples_list:
    reduced = coord_submatrix.drop(columns=sample)
    row_means = reduced.mean(axis=1)
    X = reduced.sub(row_means, axis="index")
    pca = PCA(n_components=1, random_state=0)
    pca.fit(X.T)
    leave_one_out_l1.append(pca.explained_variance_ratio_[0])

leave_one_out_l1 = np.array(leave_one_out_l1)
z_scores = (leave_one_out_l1 - leave_one_out_l1.mean()) / leave_one_out_l1.std()

for s, z in zip(samples_list, z_scores):
    print(s, round(z, 2))

sample_0 0.38
sample_1 -0.42
sample_2 0.41
sample_3 -0.17
sample_4 -4.19
sample_5 -1.79
sample_6 0.38
sample_7 1.15
sample_8 1.13
sample_9 -0.31
sample_10 0.45
sample_11 -0.17
sample_12 0.02
sample_13 -1.03
sample_14 0.59
sample_15 0.87
sample_16 0.4
sample_17 -0.62
sample_18 0.47
sample_19 0.45
sample_20 0.36
sample_21 0.47
sample_22 -1.02
sample_23 0.15
sample_24 0.24
sample_25 0.65
sample_26 -0.05
sample_27 0.14
sample_28 0.68
sample_29 0.4
